"First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์"

In [1]:
import requests
import pandas as pd
import truststore

truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations))
stations[0]                          # ดูโครงสร้างของสถานีแรกทั้งก้อน

<class 'dict'> ['stations']
181


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-28',
  'time': '13:00',
  'PM25': {'color_id': '1', 'aqi': '13', 'value': '7.9'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '20', 'value': '28'},
  'CO': {'color_id': '1', 'aqi': '3', 'value': '0.60'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '0'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '20', 'param': 'O3'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()              # ชื่อคอลัมน์ที่ซ้อนกันจะต่อด้วยจุด เช่น "<key>.<subkey>"

(181, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

1. หนึ่งแถวของ df_now คืออะไร (หนึ่งสถานี ณ เวลาล่าสุด?)
2. คอลัมน์ใดเก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา
3. ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้ง ค่าความเข้มข้น และ ค่า AQI หรือไม่ หน่วยคืออะไร
4. ค่าตัวเลขถูกเก็บเป็นข้อความ (object) หรือตัวเลข มีค่าแปลก เช่น ค่าลบ หรือ "-" แทนค่าหายหรือไม่

In [4]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 1


count    181.000000
mean      21.407182
std        7.750111
min       -1.000000
25%       16.600000
50%       21.700000
75%       26.300000
max       43.500000
Name: AQILast.PM25.value, dtype: float64

In [5]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ
explore = hourly[hourly["ts"] < "2024-01-01"]

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(0, 8) 0 สถานี
NaT → NaT
Series([], Name: count, dtype: int64)


station_id   NaN
ts           NaN
pm25         NaN
pm10         NaN
o3           NaN
co           NaN
no2          NaN
so2          NaN
dtype: float64

In [6]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

Series([], Name: pm25, dtype: float64)